Notebook 04: Model Training (EfficientNet-B0)

In [30]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

o:\Hackthons\KrishiOS\ai


In [31]:
import torch
import torch.nn as nn
import torch.optim as optim

from configs.config import *

from training.dataset import create_dataset_and_loaders
from training.model import build_model
import training.engine as engine

from training.utils import (
    set_seed,
    save_checkpoint,
    plot_training_history,
    count_trainable_parameters,
)

print("✅ Libraries Loaded")

✅ Libraries Loaded


In [35]:
set_seed(42)

print("Device :", DEVICE)

Device : cuda


In [34]:
import inspect

print(inspect.signature(engine.train_one_epoch))

(model: torch.nn.modules.module.Module, dataloader: torch.utils.data.dataloader.DataLoader, criterion: torch.nn.modules.module.Module, optimizer: torch.optim.optimizer.Optimizer, scaler: torch.amp.grad_scaler.GradScaler, device: torch.device) -> Tuple[float, float]


In [37]:
train_dataset, val_dataset, train_loader, val_loader = create_dataset_and_loaders(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print("Training Images :", len(train_dataset))
print("Validation Images :", len(val_dataset))
print("Classes :", len(train_dataset.classes))

Training Images : 43444
Validation Images : 10861
Classes : 38


In [38]:
model = build_model(
    num_classes=NUM_CLASSES,
)

model = model.to(DEVICE)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [39]:
print(f"Trainable Parameters: {count_trainable_parameters(model):,}")

Trainable Parameters: 48,678


In [40]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.2,
    patience=2,
)

scaler = torch.amp.GradScaler("cuda")

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

In [41]:
best_accuracy = 0.0

for epoch in range(EPOCHS):

    print("=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = engine.train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
    )

    val_loss, val_acc = engine.validate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.2f}%")

    if val_acc > best_accuracy:
        best_accuracy = val_acc

        save_checkpoint(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "accuracy": val_acc,
            },
            PROJECT_ROOT / "models" / "best_model.pth",
        )

        print("✅ Best model saved.")

Epoch 1/15


Train Loss : 0.5888
Train Acc  : 0.86%
Val Loss   : 0.2176
Val Acc    : 0.94%
✅ Best model saved.
Epoch 2/15


Train Loss : 0.2456
Train Acc  : 0.93%
Val Loss   : 0.1543
Val Acc    : 0.96%
✅ Best model saved.
Epoch 3/15


Train Loss : 0.2043
Train Acc  : 0.94%
Val Loss   : 0.1368
Val Acc    : 0.96%
✅ Best model saved.
Epoch 4/15


Train Loss : 0.1847
Train Acc  : 0.94%
Val Loss   : 0.1247
Val Acc    : 0.96%
✅ Best model saved.
Epoch 5/15


...

Train Loss : 0.1523
Train Acc  : 0.95%
Val Loss   : 0.0979
Val Acc    : 0.97%
✅ Best model saved.
Epoch 12/15


Train Loss : 0.1489
Train Acc  : 0.95%
Val Loss   : 0.1025
Val Acc    : 0.97%
Epoch 13/15


Train Loss : 0.1485
Train Acc  : 0.95%
Val Loss   : 0.0950
Val Acc    : 0.97%
✅ Best model saved.
Epoch 14/15


Train Loss : 0.1525
Train Acc  : 0.95%
Val Loss   : 0.0935
Val Acc    : 0.97%
Epoch 15/15


Train Loss : 0.1488
Train Acc  : 0.95%
Val Loss   : 0.0979
Val Acc    : 0.97%


In [42]:
plot_training_history(
    history,
    PROJECT_ROOT / "outputs" / "training_history.png",
)

In [43]:
print("=" * 60)
print("Training Complete")
print(f"Best Validation Accuracy : {best_accuracy:.2f}%")
print("Model Saved :", PROJECT_ROOT / "models" / "best_model.pth")
print("Plot Saved  :", PROJECT_ROOT / "outputs" / "training_history.png")

Training Complete
Best Validation Accuracy : 0.97%
Model Saved : O:\Hackthons\KrishiOS\ai\models\best_model.pth
Plot Saved  : O:\Hackthons\KrishiOS\ai\outputs\training_history.png
